# The Agent-Computer Interface (ACI)

Anthropic coined **Agent-Computer Interface (ACI)** to describe the boundary where an LLM agent meets your software: **tool schemas**, **observations**, **error strings**, and **feedback** the model must read and act on.

ACI is the agent analogue of **Human-Computer Interface (HCI)** — but the "user" is a language model with:

- A limited context window.
- No persistent memory of your codebase unless you add it.
- Brittle habits around JSON and tool selection.

Models do not "see" your Python functions. They see **serialized tool definitions** and **text observations** returned after execution. **ACI design is UX for models** — and it matters as much as model choice.

This notebook defines ACI, compares it to HCI, demonstrates **tool design for models**, **parseable error messages**, a consistent **observation format**, and a **checklist for tool authors** — all with a working **ShopPulse Order API** assistant. No prior notebooks or frameworks are required.

In [ ]:
"""
ShopPulse Order API — ACI teaching environment.
"""

import json
import os
import re
import uuid
from dataclasses import dataclass, field
from typing import Any, Callable, Literal

OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", "sk-proj-placeholder-replace-with-your-real-key")
os.environ.setdefault("OPENAI_API_KEY", OPENAI_API_KEY)


def pretty(obj: Any) -> str:
    return json.dumps(obj, indent=2, default=str)


API_DOCS: list[dict[str, str]] = [
    {"id": "auth-01", "title": "Authentication", "text": "Send X-ShopPulse-Key header on every request. Rotate keys from Dashboard > Developers."},
    {"id": "orders-02", "title": "Order lifecycle", "text": "Orders: pending → paid → shipped → delivered. Cancel only within 1 hour of payment."},
    {"id": "refunds-03", "title": "Refund policy", "text": "Full refunds within 30 days of delivery. POST /v1/orders/{id}/refunds."},
    {"id": "shipping-04", "title": "Shipping updates", "text": "PATCH /v1/orders/{id}/shipping with carrier and tracking_number."},
]

ORDERS: dict[str, dict[str, Any]] = {
    "ORD-7001": {"id": "ORD-7001", "status": "paid", "total_usd": 89.99, "customer": "alice@example.com"},
    "ORD-7002": {"id": "ORD-7002", "status": "delivered", "total_usd": 24.50, "customer": "bob@example.com"},
}

VALID_CARRIERS = ["FedEx", "UPS", "USPS", "DHL"]

print(f"Docs: {len(API_DOCS)} | Orders: {len(ORDERS)}")

---

## 1. ACI Definition

> *"The Agent-Computer Interface (ACI) is the collection of tools, APIs, and feedback mechanisms through which an agent interacts with the computer."* — Anthropic

| HCI element | ACI equivalent |
|-------------|----------------|
| Buttons, menus | **Tool schemas** (`name`, `description`, `parameters`) |
| Screen layout | **Observation strings** returned after tool execution |
| Error dialogs | **Actionable error text** in tool result messages |
| Accessibility | **Token-efficient**, unambiguous formats |
| User testing | **Eval traces** — did the model pick the right tool? |

**Key insight:** Invest in ACI as seriously as you invest in frontend UX. A brilliant model with poorly designed tools will loop, hallucinate, and waste tokens.

---

## 2. ACI vs HCI — Same Goals, Different Constraints

```
HCI                          ACI
────                         ───
Human reads labels           Model reads JSON schema + descriptions
Visual hierarchy             Flat tool list in the prompt
Muscle memory                No memory between sessions (unless added)
Forgiving typos              Strict JSON args — typos fail parsing
Rich UI feedback             Single string per tool_call_id
```

| Constraint | Implication for tool authors |
|------------|------------------------------|
| **Context window** | Keep observations concise; paginate large results |
| **No implicit state** | Return ids, counts, and next-step hints in observations |
| **Parallel tool calls** | Each observation must be self-contained |
| **Retry loops** | Errors must say *what* failed and *how to fix* |

---

## 3. Tool Design UX for Models

Good ACI starts at the **schema layer** — before any code runs:

| Principle | Bad | Good |
|-----------|-----|------|
| **Name** | `doStuff` | `search_docs` — verb + noun |
| **Description** | "Searches" | "Keyword search over API docs. Use before answering policy questions." |
| **Parameters** | `q` | `query: str` with description |
| **Enums** | free-text carrier | `enum: ["FedEx", "UPS", "USPS", "DHL"]` |
| **Scope** | one mega-tool | narrow tools the model can disambiguate |

**Rule of thumb:** If two tools differ only in an internal flag, the model will confuse them. Split by **user intent**, not by implementation.

In [ ]:
SEARCH_DOCS_SCHEMA = {
    "type": "function",
    "function": {
        "name": "search_docs",
        "description": (
            "Keyword search over ShopPulse API documentation. "
            "Use BEFORE answering policy or how-to questions. "
            "Do NOT use for live order data — use get_order instead."
        ),
        "parameters": {
            "type": "object",
            "properties": {
                "query": {"type": "string", "description": "Search phrase, e.g. 'refund policy' or 'API key'."},
                "limit": {"type": "integer", "description": "Max chunks to return (1-5). Default 3."},
            },
            "required": ["query"],
        },
    },
}

GET_ORDER_SCHEMA = {
    "type": "function",
    "function": {
        "name": "get_order",
        "description": "Fetch live order details by ID (ORD-####). Use when user mentions a specific order.",
        "parameters": {
            "type": "object",
            "properties": {
                "order_id": {"type": "string", "description": "Order ID like ORD-7001."},
            },
            "required": ["order_id"],
        },
    },
}

BAD_SCHEMA = {
    "type": "function",
    "function": {
        "name": "helper",
        "description": "Helps with stuff",
        "parameters": {"type": "object", "properties": {}},
    },
}

print("Good schema name:", SEARCH_DOCS_SCHEMA["function"]["name"])
print("Bad schema name:", BAD_SCHEMA["function"]["name"])
print("Good description length:", len(SEARCH_DOCS_SCHEMA["function"]["description"]))

---

## 4. Observation Format — What the Model Reads Back

After execution, the model sees a **tool result message**. Design a **consistent observation envelope** so the model can scan long transcripts quickly:

```
[STATUS: ok | error | empty]
[TOOL: search_docs]
[SUMMARY: one-line headline]
---
<body — facts, ids, numbers>
[HINT: optional next action]
```

| Field | Purpose |
|-------|--------|
| **STATUS** | Lets the model branch without parsing prose |
| **SUMMARY** | Fast scan in long transcripts |
| **Body** | Ground truth — cite stable ids like `[auth-01]` |
| **HINT** | Reduces wasted turns ("try get_order for live data") |

In [ ]:
ObservationStatus = Literal["ok", "error", "empty"]


def format_observation(
    tool_name: str,
    status: ObservationStatus,
    body: str,
    hint: str = "",
) -> str:
    """ACI-friendly observation envelope for tool result content."""
    summary = body.split("\n")[0][:80] if body else "(no content)"
    lines = [
        f"[STATUS: {status}]",
        f"[TOOL: {tool_name}]",
        f"[SUMMARY: {summary}]",
        "---",
        body,
    ]
    if hint:
        lines.append(f"[HINT: {hint}]")
    return "\n".join(lines)


def _search_docs_raw(query: str, limit: int = 3) -> tuple[list[dict[str, str]], str]:
    terms = [t for t in re.split(r"\W+", query.lower()) if len(t) > 2]
    scored = []
    for doc in API_DOCS:
        text = f"{doc['title']} {doc['text']}".lower()
        score = sum(1 for term in terms if term in text) if terms else 1
        scored.append((score, doc))
    scored.sort(key=lambda x: -x[0])
    hits = [d for _, d in scored[:max(1, min(limit, 5))]]
    body = "\n".join(f"[{h['id']}] {h['title']}: {h['text']}" for h in hits)
    return hits, body


hits, body = _search_docs_raw("refund policy")
print(format_observation(
    "search_docs", "ok", body,
    hint="Cite chunk ids like [refunds-03] in your final answer.",
))

---

## 5. Error Messages Models Can Parse

**Anti-pattern:** `raise Exception("fail")` → model sees opaque stack traces or empty observations.

**Pattern:** Return structured, actionable error **strings** the model can correct on the next turn:

| Error type | Model-parseable message |
|------------|-------------------------|
| Unknown order | `Unknown order_id: ORD-9999. Valid format: ORD-####. Known: ORD-7001, ORD-7002` |
| Missing arg | `Missing required field 'query'. Example: search_docs(query="refund")` |
| Auth failure | `403 Forbidden — API key missing. See [auth-01].` |
| Rate limit | `429 Rate limited. Retry after 30s or narrow query.` |
| Invalid enum | `Invalid carrier 'DHL Express'. Allowed: FedEx, UPS, USPS, DHL` |

Errors are **observations**, not exceptions that crash the agent loop.

In [ ]:
def get_order_aci(order_id: str) -> str:
    """ACI-wrapped order lookup with parseable errors."""
    oid = order_id.upper().strip()
    if not re.match(r"^ORD-\d{4}$", oid):
        return format_observation(
            "get_order", "error",
            f"Invalid order_id format: '{order_id}'. Expected ORD-#### (e.g. ORD-7001).",
            hint="Ask the user to confirm the order ID.",
        )
    order = ORDERS.get(oid)
    if not order:
        known = ", ".join(sorted(ORDERS.keys()))
        return format_observation(
            "get_order", "error",
            f"Order {oid} not found. Known orders in sandbox: {known}.",
            hint="Verify the ID with the user or search_docs for policy context.",
        )
    body = pretty(order)
    return format_observation(
        "get_order", "ok", body,
        hint=f"Order {oid} status is {order['status']}.",
    )


print(get_order_aci("ORD-9999"))
print("\n" + "=" * 40 + "\n")
print(get_order_aci("ORD-7001"))

---

## 6. Good vs Bad Tool Responses

| Scenario | Bad observation | Good observation |
|----------|-----------------|----------------|
| Empty search | `None` or `""` | `[STATUS: empty]` + hint to broaden query |
| Huge payload | 50 KB JSON dump | Paginated top-k with `[SUMMARY]` |
| Success | Raw unlabeled JSON | `[auth-01]` cited lines |
| Ambiguous error | `Error` | `Invalid order_id format. Expected ORD-####.` |

Models **retry and recover** when errors are specific. They **hallucinate** when observations are empty or vague.

In [ ]:
def search_docs_aci(query: str, limit: int = 3) -> str:
    hits, body = _search_docs_raw(query, limit)
    if not hits:
        return format_observation(
            "search_docs", "empty",
            "No documentation chunks matched your query.",
            hint="Broaden query: try 'refund', 'API key', 'shipping', or 'order lifecycle'.",
        )
    hint = "Cite [doc-id] in answers."
    if len(hits) == limit:
        hint += f" Showing top {limit} results — narrow query for fewer hits."
    return format_observation("search_docs", "ok", body, hint=hint)


print("--- Empty search (good ACI) ---")
print(search_docs_aci("kubernetes quantum"))
print("\n--- Successful search ---")
print(search_docs_aci("refund"))

---

## 7. Token Budget and Pagination

Large observations **consume context** and push out conversation history. Design ACI assuming **megabyte backends, kilobyte observations**.

| Strategy | When to use |
|----------|-------------|
| **Top-k truncation** | Search returns max N chunks |
| **Summary + expand** | Return titles first; detail tool for full text |
| **Cursor / offset** | `search_docs(query, offset=5)` for page 2 |
| **Stable ids** | `[refunds-03]` references instead of repeating full text |

In [ ]:
def search_docs_paginated(query: str, limit: int = 1, offset: int = 0) -> str:
    terms = [t for t in re.split(r"\W+", query.lower()) if len(t) > 2]
    all_hits = []
    for doc in API_DOCS:
        text = f"{doc['title']} {doc['text']}".lower()
        if not terms or any(term in text for term in terms):
            all_hits.append(doc)
    page = all_hits[offset:offset + limit]
    body = "\n".join(f"[{h['id']}] {h['text'][:80]}..." for h in page) if page else "No results on this page."
    hint = f"Page {offset // limit + 1}: showing {len(page)} of {len(all_hits)} total."
    if offset + limit < len(all_hits):
        hint += f" Use offset={offset + limit} for next page."
    status: ObservationStatus = "ok" if page else "empty"
    return format_observation("search_docs", status, body, hint=hint)


print(search_docs_paginated("order", limit=1, offset=0))
print("\n--- Page 2 ---")
print(search_docs_paginated("order", limit=1, offset=1))

---

## 8. Tool Naming and Discoverability

Models pick tools from a **flat list**. Names are navigation labels:

```
✓ search_docs      — clear action + domain
✓ get_order        — fetch by id
✓ update_shipping  — explicit write
✗ notes            — noun only — search or fetch?
✗ handle_request   — meaningless
✗ api_v2_search    — implementation leak
```

**Description duplication:** Repeat constraints in the **description** even when JSON schema enforces them — models read descriptions more reliably than `required` arrays alone.

In [ ]:
TOOL_NAME_AUDIT = {
    "search_docs": {"score": "good", "reason": "verb_noun, clear domain"},
    "get_order": {"score": "good", "reason": "verb_noun, explicit fetch"},
    "helper": {"score": "bad", "reason": "vague, not discoverable"},
    "api_v2_orders_search": {"score": "bad", "reason": "implementation detail in name"},
    "notes": {"score": "bad", "reason": "noun only, ambiguous intent"},
}

print(f"{'Tool name':<25} {'Score':<6} Reason")
print("-" * 60)
for name, info in TOOL_NAME_AUDIT.items():
    print(f"{name:<25} {info['score']:<6} {info['reason']}")

---

## 9. ACI Across Provider Protocols

ACI sits **above** the wire protocol — it is how you design **semantics** regardless of vendor:

| Layer | Your responsibility | Provider handles |
|-------|---------------------|------------------|
| Schema design | Names, descriptions, types | JSON schema transport |
| Tool call | Valid arguments | `tool_calls` / `tool_use` parsing |
| Observation | `format_observation` envelope | `role: tool` message delivery |
| Errors | Actionable strings | Invalid call detection |

OpenAI uses `tool_calls` on assistant messages. Anthropic uses `tool_use` content blocks. Google uses `functionCall`. The **observation format you design** can be identical across all three.

In [ ]:
# Simulated round-trip — provider-agnostic message shapes
simulated_assistant = {
    "role": "assistant",
    "content": None,
    "tool_calls": [{
        "id": "call_aci_demo",
        "type": "function",
        "function": {"name": "search_docs", "arguments": json.dumps({"query": "refund"})},
    }],
}

observation = search_docs_aci("refund")
tool_message = {
    "role": "tool",
    "tool_call_id": "call_aci_demo",
    "content": observation,
}

print("Assistant emitted:", simulated_assistant["tool_calls"][0]["function"]["name"])
print("\nTool result preview:")
print(tool_message["content"][:300], "...")

---

## 10. ACI-Wrapped Tool Registry

Wrap every tool implementation with consistent observation formatting at the registry boundary — the single ACI exit point.

In [ ]:
ACI_TOOLS: dict[str, Callable[..., str]] = {
    "search_docs": search_docs_aci,
    "get_order": get_order_aci,
}

TOOL_SCHEMAS = [SEARCH_DOCS_SCHEMA, GET_ORDER_SCHEMA]


@dataclass
class ACIToolExecutor:
    """Dispatches tool calls and always returns ACI-formatted observations."""

    registry: dict[str, Callable[..., str]] = field(default_factory=lambda: dict(ACI_TOOLS))
    trace: list[dict[str, str]] = field(default_factory=list)

    def execute(self, tool_name: str, args: dict[str, Any], tool_call_id: str) -> dict[str, str]:
        fn = self.registry.get(tool_name)
        if fn is None:
            content = format_observation(
                tool_name, "error",
                f"Unknown tool: {tool_name}. Available: {', '.join(self.registry.keys())}.",
            )
        else:
            try:
                content = fn(**args)
            except Exception as exc:
                content = format_observation(
                    tool_name, "error",
                    f"Execution failed: {exc}",
                    hint="Check arguments and retry.",
                )
        self.trace.append({"tool": tool_name, "status": content.split("\n")[0]})
        return {"role": "tool", "tool_call_id": tool_call_id, "content": content}


aci_executor = ACIToolExecutor()
for call in [
    ("search_docs", {"query": "API key"}, "call_1"),
    ("get_order", {"order_id": "ORD-7002"}, "call_2"),
    ("get_order", {"order_id": "bad-id"}, "call_3"),
]:
    msg = aci_executor.execute(*call)
    print(f"[{call[0]}] {msg['content'].split(chr(10))[0]}")
print(f"\nTrace: {aci_executor.trace}")

---

## 11. ACI Checklist for Tool Authors

Use this before shipping any new tool. The validator below checks what can be automated; the rest is human review.

In [ ]:
ACI_CHECKLIST = [
    "verb_noun tool name, unique in registry",
    "description explains WHEN to call the tool",
    "description explains WHEN NOT to call (if ambiguous)",
    "all parameters typed and described",
    "success observations use STATUS/SUMMARY envelope",
    "errors include valid values and fix hints",
    "no secrets or stack traces in observations",
    "large results paginated or summarized",
    "read vs write side effects documented",
    "3+ eval prompts cover realistic usage",
]


def validate_schema_aci(schema: dict[str, Any]) -> list[str]:
    """Automated ACI checks on a tool schema. Returns list of issues."""
    issues = []
    fn = schema.get("function", {})
    name = fn.get("name", "")
    desc = fn.get("description", "")
    params = fn.get("parameters", {}).get("properties", {})

    if not name or "_" not in name and name in ("helper", "notes", "api"):
        issues.append(f"name '{name}' is not verb_noun or is too vague")
    if len(desc) < 40:
        issues.append("description too short — explain WHEN to use")
    for pname, pdef in params.items():
        if "description" not in pdef:
            issues.append(f"parameter '{pname}' missing description")
    return issues


for schema in [SEARCH_DOCS_SCHEMA, BAD_SCHEMA]:
    name = schema["function"]["name"]
    issues = validate_schema_aci(schema)
    status = "PASS" if not issues else "FAIL"
    print(f"{name}: {status}")
    for issue in issues:
        print(f"  - {issue}")

print(f"\nFull checklist ({len(ACI_CHECKLIST)} items):")
for i, item in enumerate(ACI_CHECKLIST, 1):
    print(f"  {i:2}. [ ] {item}")

---

## 12. ACI in the Agent Loop

```
User question
     │
     ▼
┌─────────┐    tool schemas (ACI input)
│   LLM   │◄──────────────────────────────┐
└────┬────┘                               │
     │ tool_calls                         │ tool result message
     ▼                                    │ (ACI output)
┌─────────┐                               │
│  Tools  │───────────────────────────────┘
└─────────┘
```

Every loop iteration is an ACI round trip. Poor observations → more iterations → higher cost and latency.

In [ ]:
class OfflineACIPlanner:
    """Simulates LLM tool selection for ACI demo."""

    def plan(self, user_text: str, messages: list[dict[str, Any]]) -> dict[str, Any]:
        if any(m.get("role") == "tool" for m in messages):
            last = [m for m in messages if m.get("role") == "tool"][-1]["content"]
            return {"role": "assistant", "content": f"Based on tool result:\n{last[:200]}..."}
        t = user_text.lower()
        if re.search(r"ord-\d+", t, re.I):
            oid = re.search(r"ord-\d+", t, re.I).group(0).upper()
            return {"role": "assistant", "tool_calls": [{"id": "call_1", "name": "get_order", "args": {"order_id": oid}}]}
        if any(w in t for w in ("policy", "how", "api key", "refund rules", "shipping")):
            return {"role": "assistant", "tool_calls": [{"id": "call_1", "name": "search_docs", "args": {"query": user_text}}]}
        return {"role": "assistant", "content": "Ask about ShopPulse orders (ORD-####) or API documentation."}


@dataclass
class ACIAgent:
    executor: ACIToolExecutor = field(default_factory=ACIToolExecutor)
    planner: OfflineACIPlanner = field(default_factory=OfflineACIPlanner)
    messages: list[dict[str, Any]] = field(default_factory=list)

    def run(self, user_message: str) -> str:
        self.messages = [
            {"role": "system", "content": "ShopPulse assistant. Tools return ACI-formatted observations."},
            {"role": "user", "content": user_message},
        ]
        for _ in range(3):
            decision = self.planner.plan(user_message, self.messages)
            self.messages.append(decision)
            if decision.get("content") and not decision.get("tool_calls"):
                return decision["content"]
            for tc in decision.get("tool_calls", []):
                tool_msg = self.executor.execute(tc["name"], tc["args"], tc["id"])
                self.messages.append(tool_msg)
        return "Max rounds reached."


for query in [
    "What is the refund policy?",
    "Status of order ORD-7001",
    "Lookup order ORD-9999",
]:
    agent = ACIAgent()
    print(f"Q: {query}")
    print(f"A: {agent.run(query)[:150]}...\n")

---

## 13. Common ACI Anti-Patterns

| Anti-pattern | Symptom | Fix |
|--------------|---------|-----|
| God tool | Wrong args, hallucinated fields | Split into narrow tools |
| Leaky errors | Model repeats stack traces to user | Catch and `format_observation` |
| Implicit context | "Updated it" with no id | Return `order_id=ORD-7001` |
| Duplicate tools | `fetch_order` + `get_order` | One canonical name |
| HTML / XML blobs | Model ignores structure | Plain text + `[STATUS]` markers |
| Non-idempotent writes | Double-create on retry | Idempotency keys |
| Empty observations | Model hallucinates success | Always return STATUS envelope |

---

## 14. Measuring ACI Quality

Track these metrics in evaluation runs:

| Metric | How to measure |
|--------|----------------|
| **Tool selection accuracy** | Correct tool chosen for prompt |
| **Arg validity rate** | First-call JSON passes schema |
| **Recovery rate** | Model fixes after parseable error |
| **Avg tool turns** | Lower is cheaper (if task succeeds) |
| **Observation token cost** | Characters returned per tool call |

In [ ]:
EVAL_PROMPTS = [
    ("What is the refund policy?", "search_docs"),
    ("Status of ORD-7002", "get_order"),
    ("How do API keys work?", "search_docs"),
    ("Lookup ORD-9999", "get_order"),  # should get parseable error, then recover
]


def observation_token_cost(content: str) -> int:
    return len(content)  # rough proxy; divide by 4 for token estimate


print(f"{'Prompt':<35} {'Expected tool':<15} Est. tokens (obs)")
print("-" * 65)
for prompt, expected in EVAL_PROMPTS:
    agent = ACIAgent()
    agent.run(prompt)
    tool_msgs = [m for m in agent.messages if m.get("role") == "tool"]
    obs_cost = sum(observation_token_cost(m["content"]) for m in tool_msgs)
    print(f"{prompt:<35} {expected:<15} ~{obs_cost // 4}")

---

## 15. Tool Catalog for Engineers

Models read JSON schemas. **Engineers** read internal catalogs. Maintain both:

```markdown
## search_docs
- **When:** User asks about ShopPulse API behavior, policies, or setup
- **When NOT:** Live order lookups — use get_order
- **Returns:** [doc-id] prefixed lines
- **Errors:** STATUS empty → broaden query

## get_order
- **When:** User mentions ORD-#### or asks for order status
- **When NOT:** Policy questions — use search_docs
- **Returns:** JSON order object with STATUS ok
- **Errors:** Invalid format or unknown ID with valid examples
```

---

## 16. Versioning Tool Schemas

Breaking changes confuse routing habits accumulated in evals and production traces:

| Change type | Strategy |
|-------------|----------|
| Rename tool | Deprecate old name for 2 releases |
| New required field | Default value + migration period |
| Observation format | Version prefix `[v2]` in SUMMARY |
| Split god tool | Add new tools; keep shim that delegates |

Log `tool_schema_version` in traces for debugging.

In [ ]:
TOOL_CATALOG = {
    "search_docs": {
        "version": "1.2.0",
        "when": "Policy and how-to questions",
        "when_not": "Live order data",
        "observation_format": "ACI envelope v1",
    },
    "get_order": {
        "version": "1.0.0",
        "when": "User mentions ORD-####",
        "when_not": "Documentation search",
        "observation_format": "ACI envelope v1",
    },
}

print(pretty(TOOL_CATALOG))

---

## 17. Check Your Understanding

1. What is the Agent-Computer Interface, and how does it relate to HCI?
2. What are the four fields in the `format_observation` envelope?
3. Why should error messages include valid values and hints?
4. What is wrong with a tool named `helper` with description "Helps with stuff"?
5. How does pagination protect the context window?
6. What does `validate_schema_aci` check automatically vs the human checklist?
7. Why log `tool_schema_version` in production traces?

---

## 18. Summary

**ACI** is the UX layer between your application and the model. The model only experiences **schemas** and **observation strings** — design both deliberately.

**Key takeaways:**

- **Tool schemas** are navigation labels: verb_noun names, rich descriptions, typed parameters.
- **Observations** should use a consistent `[STATUS]` / `[SUMMARY]` / body / `[HINT]` envelope.
- **Errors** are observations too — include valid values, examples, and recovery hints.
- **Paginate and summarize** large results; assume megabyte backends and kilobyte observations.
- **One registry boundary** (`ACIToolExecutor`) ensures every tool returns ACI-formatted output.
- **Measure** tool selection accuracy, arg validity, recovery rate, and observation token cost.
- **Version** schemas and maintain an engineer-facing tool catalog alongside JSON definitions.

Invest in ACI as seriously as frontend design — it determines whether your agents complete tasks in one turn or five.